In [1]:
import os
from Database import *
import pandas as pd
import requests

In [2]:
folder = 'news'
files = os.listdir(folder)
tickers = [f.split("_")[0] for f in files if os.path.isfile(os.path.join(folder, f))]

print(tickers)

['BA', 'MCD', 'JNJ', 'INTC', 'BAC', 'AMZN', 'NVDA', 'ORCL', 'DIS', 'GS', 'GOOG', 'MSFT', 'TSLA', 'META', 'AAPL', 'V', 'PFE', 'KO', 'GE', 'CAT', 'XOM', 'CVX', 'DE', 'F', 'WMT']


In [3]:
class AlphavantageOHLCVScrapper:
    def __init__(self) -> None:
        self.api_key = os.getenv('ALPHAVANTAGE_KEY')
        
    def fetch_ohlcv_data(self, ticker:str, interval:str, month:str, outputsize:str='full'):
        url = f"https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol={ticker}&interval={interval}&apikey={self.api_key}&month={month}&outputsize={outputsize}"
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                if 'Time Series (1min)' in data:
                    return data['Time Series (1min)']
                else:
                    print(data)
        except Exception as error:
            print(f"Error in fetching data: {error}")
            return None
        
    def process_data_as_df(self,data, ticker:str):
        df = pd.DataFrame.from_dict(data, orient='index')
        df.columns = [col + f"_{ticker}" for col in ['open', 'high', 'low', 'close', 'volume']]
        df.index = pd.to_datetime(df.index)
        df = df.iloc[::-1]
        return df
    
    def create_ohlcv_df(self, ticker, open_time, close_time):
        months = pd.date_range(start=open_time, end=close_time, freq='MS').strftime('%Y-%m').tolist()
        dfs_to_concat = []
        for month in months:
            data = self.fetch_ohlcv_data(ticker, '1min', month)
            if data:
                df = self.process_data_as_df(data, ticker)
                dfs_to_concat.append(df)
        df_ohlcv = pd.concat(dfs_to_concat)
        return df_ohlcv
    
sc = AlphavantageOHLCVScrapper()

    

In [17]:
tickers = [  
#'V', 
#'PFE', 
#'KO', 
#'GE', 

#'CAT', 
#'XOM', 
#'CVX', 
#'DE', 

#'F', 
#'WMT'
]


open_time = pd.to_datetime('2024-04-01')
close_time = pd.to_datetime('2024-09-01')


for ticker in tickers:
    print(f"Fetching data for {ticker}")
    df = sc.create_ohlcv_df(ticker, open_time, close_time)
    df.to_csv(f'ohlcv/{ticker}_ohlcv.csv')
    print(f"Data for {ticker} fetched and saved with shape: {df.shape}")
    display(df.head())
    display(df.tail())
    break

Fetching data for WMT
Data for WMT fetched and saved with shape: (91863, 5)


,open_WMT,high_WMT,low_WMT,close_WMT,volume_WMT
2024-04-01 04:00:00,59.8350,59.8449,59.7952,59.8449,162
2024-04-01 04:01:00,59.8151,59.8747,59.8151,59.8747,142
2024-04-01 04:07:00,59.8747,59.8747,59.8548,59.8548,8
2024-04-01 04:08:00,59.8946,59.8946,59.8946,59.8946,1
2024-04-01 04:09:00,59.9145,59.9741,59.9045,59.9045,105


,open_WMT,high_WMT,low_WMT,close_WMT,volume_WMT
2024-09-30 19:54:00,80.6800,80.6800,80.6800,80.6800,5
2024-09-30 19:55:00,80.7200,80.7200,80.7200,80.7200,1
2024-09-30 19:56:00,80.7600,80.7600,80.7600,80.7600,1
2024-09-30 19:58:00,80.7400,80.7400,80.7200,80.7200,21
2024-09-30 19:59:00,80.7400,80.7400,80.7000,80.7400,200
